In [17]:
%pip install requests beautifulsoup4 pdfkit

Note: you may need to restart the kernel to use updated packages.


In [18]:
import os
import re
import requests
import pdfkit
from bs4 import BeautifulSoup
from pathlib import Path

In [ ]:
def get_books_and_links(toc_url):
    response = requests.get(toc_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    books = {}
    current_book = None

    tags = soup.find_all(["h2", "ul"])
    i = 0
    while i < len(tags):
        tag = tags[i]

        if tag.name == "h2" and "Book" in tag.get_text():
            current_book = tag.get_text(strip=True)
            books[current_book] = []

            if i + 1 < len(tags) and tags[i + 1].name == "ul":
                ul = tags[i + 1]

                chapter_lis = ul.find_all("li", recursive=False)
                if len(chapter_lis) == 1 and chapter_lis[0].find("ul"):
                    chapter_lis = chapter_lis[0].find("ul").find_all("li", recursive=False)

                for li in chapter_lis:
                    a = li.find("a", href=True)
                    if a:
                        title = a.get_text(strip=True)
                        href = a["href"]
                        books[current_book].append((title, href))

                i += 1  # Skip the UL

        i += 1

    return books

books = get_books_and_links("https://practicalguidetoevil.wordpress.com/table-of-contents/")

# Preview result
for book, chapters in books.items():
    print(f"\n📘 {book} ({len(chapters)} chapters)")
    for title, _ in chapters:  # Show first 3 chapters
        print(f"  - {title}")

TypeError: get_books_and_links() missing 1 required positional argument: 'toc_url'

In [7]:
def sanitize_filename(title, index):
    title = re.sub(r'[^\w\s-]', '', title)  # Remove special characters
    title = re.sub(r'\s+', '_', title.strip())  # Replace spaces with underscores
    return f"Chapter_{index:02d}_{title}.pdf"

In [9]:
async def save_books_as_pdfs(books):
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        context = await browser.new_context()

        for book_title, chapters in books.items():
            print(f"\n📘 Saving {book_title}...")

            book_folder = output_dir / book_title.replace(" ", "_")
            book_folder.mkdir(exist_ok=True)

            for i, (chapter_title, url) in enumerate(chapters, start=1):
                file_name = sanitize_filename(chapter_title, i)
                file_path = book_folder / file_name

                print(f"  - Saving: {chapter_title} → {file_name}")
                try:
                    page = await context.new_page()
                    await page.goto(url, wait_until="networkidle")
                    await page.pdf(path=str(file_path), format="A4")
                    await page.close()
                except Exception as e:
                    print(f"    ⚠️ Failed to save {chapter_title}: {e}")

        await browser.close()

In [13]:
# Run the async function from Jupyter
await save_books_as_pdfs({k: v for k, v in books.items() if k == "Book 1"})

NotImplementedError: 